# **Cultura e Práticas em DataOps e MLOps**
**Autor**: Renan Santos Mendes

**Email**: renansantosmendes@gmail.com

**Descrição**: Este notebook apresenta um exemplo de uma rede neural profunda com mais de uma camada para um problema de classificação.


# **Saúde Fetal**

As Cardiotocografias (CTGs) são opções simples e de baixo custo para avaliar a saúde fetal, permitindo que os profissionais de saúde atuem na prevenção da mortalidade infantil e materna. O próprio equipamento funciona enviando pulsos de ultrassom e lendo sua resposta, lançando luz sobre a frequência cardíaca fetal (FCF), movimentos fetais, contrações uterinas e muito mais.

Este conjunto de dados contém 2126 registros de características extraídas de exames de Cardiotocografias, que foram então classificados por três obstetras especialistas em 3 classes:

- Normal
- Suspeito
- Patológico

# Instalando pacotes

In [ ]:
!pip install mlflow

# 1 - Importando os módulos necessários

In [ ]:
import os
import random
import numpy as np
import random as python_random
import tensorflow
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, InputLayer
from keras.utils import to_categorical

import pandas as pd
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Definindo funções adicionais

In [ ]:
def reset_seeds() -> None:
  """
  Resets the seeds to ensure reproducibility of results.

  This function sets the seed for various random number generation libraries
  to ensure that results are reproducible. The affected libraries are:
  - Python's built-in `random`
  - NumPy
  - TensorFlow

  The seed used is 42.

  Returns:
      None
  """
  os.environ['PYTHONHASHSEED']=str(42)
  tf.random.set_seed(42)
  np.random.seed(42)
  random.seed(42)

# 2 - Fazendo a leitura do dataset e atribuindo às respectivas variáveis

In [ ]:
url = 'raw.githubusercontent.com'
username = 'renansantosmendes'
repository = 'lectures-cdas-2023'
file_name = 'fetal_health_reduced.csv'
data = pd.read_csv(f'https://{url}/{username}/{repository}/master/{file_name}')

# Dando uma leve olhada nos dados

In [ ]:
data.head()

,severe_decelerations,accelerations,fetal_movement,uterine_contractions,fetal_health
0,0.0,0.0,0.0,0.0,2.0
1,0.0,6.0,0.0,6.0,1.0
2,0.0,3.0,0.0,8.0,1.0
3,0.0,3.0,0.0,8.0,1.0
4,0.0,7.0,0.0,8.0,1.0


# 3 - Preparando o dado antes de iniciar o treino do modelo

In [ ]:
X=data.drop(["fetal_health"], axis=1)
y=data["fetal_health"]

columns_names = list(X.columns)
scaler = preprocessing.StandardScaler()
X_df = scaler.fit_transform(X)
X_df = pd.DataFrame(X_df, columns=columns_names)

X_train, X_test, y_train, y_test = train_test_split(X_df,
                                                    y,
                                                    test_size=0.3,
                                                    random_state=42)

y_train = y_train -1
y_test = y_test - 1

# 4 - Criando o modelo e adicionando as camadas

In [ ]:
reset_seeds()
model = Sequential()
model.add(InputLayer(input_shape=(X_train.shape[1], )))
model.add(Dense(units=10, activation='relu'))
model.add(Dense(units=10, activation='relu'))
model.add(Dense(units=3, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


# 5 - Compilando o modelo


In [ ]:
model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

##**Configurando o mlflow**

In [ ]:
import mlflow

os.environ['MLFLOW_TRACKING_USERNAME'] = 'renansantosmendes'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '6d730ef4a90b1caf28fbb01e5748f0874fda6077'
mlflow.set_tracking_uri('https://dagshub.com/renansantosmendes/puc_lectures_mlops.mlflow')

# Desativa completamente o autolog
mlflow.keras.autolog(disable=True)

# 6 - Executando o treino do modelo

In [ ]:
with mlflow.start_run(run_name='experiment_mlops_ead_nik') as run:
    run_id = run.info.run_id
    print(f"Run ID: {run_id}")

    mlflow.log_params({
        "epochs": 50,
        "optimizer": "adam",
        "loss_function": "sparse_categorical_crossentropy",
        "validation_split": 0.2,
        "layers": "10-10-3",
        "activation": "relu"
    })

    history = model.fit(X_train, y_train,
                        epochs=50,
                        validation_split=0.2,
                        verbose=2)

    for epoch in range(len(history.history['loss'])):
        mlflow.log_metrics({
            "loss":         history.history['loss'][epoch],
            "val_loss":     history.history['val_loss'][epoch],
            "accuracy":     history.history['accuracy'][epoch],
            "val_accuracy": history.history['val_accuracy'][epoch],
        }, step=epoch)

    model.save("modelo.keras")
    mlflow.log_artifact("modelo.keras", artifact_path="model")
    print("Artefato salvo!")

Run ID: d1ca2dc45d1e4d908ff7533227ff0871
Epoch 1/50
38/38 - 0s - 7ms/step - accuracy: 0.8160 - loss: 0.4520 - val_accuracy: 0.8389 - val_loss: 0.4661
Epoch 2/50
38/38 - 0s - 7ms/step - accuracy: 0.8151 - loss: 0.4522 - val_accuracy: 0.8389 - val_loss: 0.4664
Epoch 3/50
38/38 - 0s - 7ms/step - accuracy: 0.8160 - loss: 0.4520 - val_accuracy: 0.8389 - val_loss: 0.4661
Epoch 4/50
38/38 - 0s - 7ms/step - accuracy: 0.8168 - loss: 0.4518 - val_accuracy: 0.8389 - val_loss: 0.4661
Epoch 5/50
38/38 - 0s - 8ms/step - accuracy: 0.8168 - loss: 0.4520 - val_accuracy: 0.8389 - val_loss: 0.4667
Epoch 6/50
38/38 - 0s - 5ms/step - accuracy: 0.8168 - loss: 0.4518 - val_accuracy: 0.8389 - val_loss: 0.4668
Epoch 7/50
38/38 - 0s - 4ms/step - accuracy: 0.8168 - loss: 0.4518 - val_accuracy: 0.8389 - val_loss: 0.4669
Epoch 8/50
38/38 - 0s - 4ms/step - accuracy: 0.8160 - loss: 0.4518 - val_accuracy: 0.8389 - val_loss: 0.4666
Epoch 9/50
38/38 - 0s - 4ms/step - accuracy: 0.8168 - loss: 0.4517 - val_accuracy: 0.83

In [ ]:
from mlflow.tracking import MlflowClient
client = MlflowClient()

version = client.create_model_version(
    name="fetal_health_nik",
    source=f"runs:/{run_id}/model/modelo.keras",
    run_id=run_id
)
print(f"Versão registrada: v{version.version}")

2026/03/24 23:39:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fetal_health_nik, version 2


Versão registrada: v2
